# Penalty caps from a revert-rate target

Two caps in bps of order size — **correlated** (USD-pegged vs USD-pegged) and **uncorrelated** —
as a function of the exclusivity window `T`, so each chain reads off the cap for its own window.

A settlement reverts on price grounds when the pair price moves against the solver by more than
the cap over the `T` seconds to the deadline, so a pair's price-driven revert rate at cap `c` is
the share of historical `T`-second moves below `-c`, averaged over both trade directions. Each
tier's cap is the smallest `c` whose flow-weighted rate is at or below `TARGET_REVERT_RATE`.

`unc_max` / `corr_max` is the max across the months measured — the column to read if the cap has
to hold in the worst of them.

In [ ]:
import os
import time
import urllib.error
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

TARGET_REVERT_RATE = 0.08       # tolerated price-driven revert rate per settlement attempt
T_REFERENCE = 26                # ethereum's window; section 2 fixes T here

# --- fit and validation periods ----------------------------------------------------------
FIT_MONTH = "2026-07"
MONTHS = {"2026-05": ("2026-05-01", "2026-05-31"),
          "2026-06": ("2026-06-01", "2026-06-30"),
          "2026-07": ("2026-07-01", "2026-07-30")}
VALIDATION_MONTHS = [m for m in sorted(MONTHS) if m != FIT_MONTH]
DAYS = {m: [d.strftime("%Y-%m-%d") for d in pd.date_range(a, b)]
        for m, (a, b) in MONTHS.items()}

# --- exclusivity windows the cap is fitted for -------------------------------------------
# (deadline_blocks - 0.5) x block_time, floored, except ethereum which is set directly.
CHAIN_WINDOWS = {
    "ethereum": 26,
    "gnosis": int((3 - 0.5) * 5),
    "arbitrum": int((11 - 0.5) * 0.25),
    "base": int((4 - 0.5) * 2),
    "polygon": int((8 - 0.5) * 1.5),
    "avalanche_c": int((8 - 0.5) * 1),
    "bnb": int((5 - 0.5) * 0.45),
    "linea": int((4 - 0.5) * 2),
    "ink": int((5 - 0.5) * 1),
    "plasma": int((5 - 0.5) * 1),
}
T_GRID = sorted(set(range(1, 31)) | set(CHAIN_WINDOWS.values()))
assert all(isinstance(T, int) and T >= 1 for T in T_GRID), \
    "windows must be whole seconds >= 1: 1-second klines cannot resolve less"

# The ten most-traded CoW pairs plus the correlated tier, with their settlement-attempt counts
# (Jan-Jun 2026 extracts). Trade direction is dropped and USD-pegged stables are
# merged into one "USD" leg, whose price series is the constant 1 -- so ("ETH", "USD") is ETHUSDT.
PAIRS = [
    # (a, b, tier, attempts)
    ("ETH",  "USD", "uncorrelated", 176228),
    ("BTC",  "USD", "uncorrelated",  83108),
    ("BNB",  "USD", "uncorrelated",  18354),
    ("BTC",  "ETH", "uncorrelated",   8739),
    ("XAUT", "USD", "uncorrelated",   8411),
    ("GNO",  "ETH", "uncorrelated",   6291),
    ("AAVE", "USD", "uncorrelated",   5233),
    ("SKY",  "USD", "uncorrelated",   5139),
    ("LINK", "USD", "uncorrelated",   4727),
    ("TAO",  "ETH", "uncorrelated",   4206),
    ("USDC", "USD", "correlated",   112494),
]

KLINE_CACHE = "../data/binance_klines_1s"
os.makedirs(KLINE_CACHE, exist_ok=True)

pairs = pd.DataFrame(PAIRS, columns=["a", "b", "group", "attempts"])
pairs["pair"] = pairs.a + "↔" + pairs.b
pairs["weight"] = pairs.attempts / pairs.groupby("group").attempts.transform("sum")
GROUPS = ["uncorrelated", "correlated"]
COLORS = {"uncorrelated": "C1", "correlated": "C0"}
BOOKS = sorted({x + "USDT" for x in [*pairs.a, *pairs.b] if x != "USD"})
print(f"{len(pairs)} pairs, {pairs.attempts.sum():,} attempts, {len(BOOKS)} books")

## 1. Price moves and cap(T)

First run downloads a few GB into `data/binance_klines_1s/`.

In [ ]:
CAP_GRID = np.concatenate([np.arange(0.0, 10.0, 0.01), np.arange(10.0, 60.0, 0.05)])


def closes_1s(symbol, day):
    """Close price per second of one UTC day, gap-filled onto the 86400-second grid."""
    path = os.path.join(KLINE_CACHE, f"{symbol}-1s-{day}.zip")
    if os.path.exists(path + ".missing"):
        return None
    url = f"https://data.binance.vision/data/spot/daily/klines/{symbol}/1s/{symbol}-1s-{day}.zip"
    for attempt in range(1, 4):
        if not os.path.exists(path):
            try:
                urllib.request.urlretrieve(url, path)
            except urllib.error.HTTPError:      # no archive for this day: permanent
                open(path + ".missing", "w").close()
                return None
            except (urllib.error.ContentTooShortError, urllib.error.URLError):
                if os.path.exists(path):       # urlretrieve leaves the partial file behind
                    os.remove(path)
                if attempt == 3:
                    raise
                time.sleep(attempt)
                continue
        try:
            with zipfile.ZipFile(path) as z:
                df = pd.read_csv(z.open(z.namelist()[0]), header=None, usecols=[0, 4],
                                 names=["ts", "close"])
            break
        except zipfile.BadZipFile:              # truncated download cached from before
            os.remove(path)
            if attempt == 3:
                raise
            time.sleep(attempt)
    if not str(df.ts.iloc[0]).isdigit():                    # newer files ship a header row
        df = df.iloc[1:].astype({"ts": np.int64, "close": float})
    unit = 1_000_000 if df.ts.iloc[0] > 10**14 else 1_000   # timestamps: us (2025+) or ms
    sec_of_day = (df.ts.to_numpy(np.int64) // unit) % 86400
    px = np.full(86400, np.nan)
    px[sec_of_day] = df.close.to_numpy(float)
    return pd.Series(px).ffill().bfill().to_numpy()


def t_moves(grid, T):
    p = grid[::T]
    return (p[1:] / p[:-1] - 1.0) * 1e4     # bps over one non-overlapping window


ONES = np.ones(86400)

GRIDS = {}
for month, days in DAYS.items():
    for day in days:
        for b in BOOKS:
            g = closes_1s(b, day)
            if g is not None:
                GRIDS[(day, b)] = g
    print(f"{month}: {len(days)} days", flush=True)
print(f"{len(GRIDS)} (day, book) series held, "
      f"{sum(g.nbytes for g in GRIDS.values()) / 1e6:.0f} MB")


def day_ratio(row, day):
    """The pair's 1-second price series for one day (a in units of b), or None if missing."""
    ga = ONES if row.a == "USD" else GRIDS.get((day, row.a + "USDT"))
    gb = ONES if row.b == "USD" else GRIDS.get((day, row.b + "USDT"))
    return None if (ga is None or gb is None) else ga / gb

In [ ]:
def revert_curve(moves):
    """Share of moves below -c, for every c in CAP_GRID."""
    return np.searchsorted(np.sort(moves), -CAP_GRID, side="left") / len(moves)


def pair_curve(row, month, T):
    """Revert rate for every c in CAP_GRID, averaging the pair's two directions equally."""
    fwd, rev = [], []
    for day in DAYS[month]:
        ratio = day_ratio(row, day)
        if ratio is None:
            continue
        fwd.append(t_moves(ratio, T))
        rev.append(t_moves(1.0 / ratio, T))
    if not fwd:
        raise ValueError(f"no price data for {row.pair} in {month}")
    return 0.5 * (revert_curve(np.concatenate(fwd)) + revert_curve(np.concatenate(rev)))


def tier_curve(month, T, group):
    """Flow-weighted revert-rate curve of a tier, over CAP_GRID."""
    rows = pairs[pairs.group == group]
    return np.sum([w * pair_curve(r, month, T)
                   for w, r in zip(rows.weight, rows.itertuples())], axis=0)


def tier_rate(month, T, group, cap):
    return float(np.interp(cap, CAP_GRID, tier_curve(month, T, group)))


def tier_cap(month, T, group):
    """Smallest cap on CAP_GRID whose flow-weighted tier rate is <= the target."""
    curve = tier_curve(month, T, group)
    i = int(np.argmax(curve <= TARGET_REVERT_RATE))
    # argmax on an all-False array returns 0, which would read as a 0 bps cap
    return CAP_GRID[i] if curve[i] <= TARGET_REVERT_RATE else np.nan


CAPS = {(m, T, g): tier_cap(m, T, g) for m in MONTHS for T in T_GRID for g in GROUPS}


def cap_max(T, g):
    return np.nanmax([CAPS[(m, T, g)] for m in MONTHS])


cap_tbl = pd.DataFrame({
    "T_s": T_GRID,
    **{f"unc_{m[-2:]}": [CAPS[(m, T, "uncorrelated")] for T in T_GRID] for m in sorted(MONTHS)},
    "unc_max": [cap_max(T, "uncorrelated") for T in T_GRID],
    "corr_max": [cap_max(T, "correlated") for T in T_GRID],
    "chains": [", ".join(sorted(c for c, t in CHAIN_WINDOWS.items() if t == T))
               for T in T_GRID],
}).set_index("T_s")
print(f"cap in bps at a {TARGET_REVERT_RATE:.0%} price-driven revert rate "
      f"(unc_NN = uncorrelated fitted on month NN)\n")
print(cap_tbl.to_string(float_format=lambda v: f"{v:8.2f}"))

In [ ]:
_, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, group in zip(axes, GROUPS):
    for month in sorted(MONTHS):
        ax.plot(T_GRID, [CAPS[(month, T, group)] for T in T_GRID], "o-", ms=3, label=month)
    ax.plot(T_GRID, [cap_max(T, group) for T in T_GRID], "k--", lw=1.2, label="max of months")
    anchor = CAPS[(FIT_MONTH, T_REFERENCE, group)]
    ax.plot(T_GRID, [anchor * np.sqrt(T / T_REFERENCE) for T in T_GRID], ":", color="gray",
            label=r"$\propto\sqrt{T}$")
    ax.set(xlabel="exclusivity window T (s)", ylabel="cap (bps of order size)",
           title=f"{group}: cap(T)")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. What a constant cap costs

The cap is one number per tier, held fixed.

- **over time** — the realized rate drifts away from the target. The table gives it per month,
  the figure per day.
- **across token pairs** — one cap serves a whole tier, so individual pairs sit above or below it.

In [ ]:
T = T_REFERENCE
fit_caps = {g: CAPS[(FIT_MONTH, T, g)] for g in GROUPS}
max_caps = {g: cap_max(T, g) for g in GROUPS}
print(f"T = {T}s, target {TARGET_REVERT_RATE:.0%}; fit {FIT_MONTH}, "
      f"validation {', '.join(VALIDATION_MONTHS)}")
print("fitted caps  : " + ", ".join(f"{g} {v:.2f}" for g, v in fit_caps.items()))
print("max-of-months: " + ", ".join(f"{g} {v:.2f}" for g, v in max_caps.items()) + "\n")
print(pd.DataFrame([
    {"group": g, "month": m, "role": "fit" if m == FIT_MONTH else "validation",
     "rate_at_fit_cap": tier_rate(m, T, g, fit_caps[g]),
     "rate_at_max_cap": tier_rate(m, T, g, max_caps[g])}
    for g in GROUPS for m in sorted(MONTHS)]).to_string(
        index=False, float_format=lambda v: f"{v:7.2%}"))

In [ ]:
rows_out = []
for month, days in DAYS.items():
    for day in days:
        for g in GROUPS:
            thr, num, den = -fit_caps[g], 0.0, 0.0
            for r in pairs[pairs.group == g].itertuples():
                ratio = day_ratio(r, day)
                if ratio is None:
                    continue
                num += r.weight * 0.5 * ((t_moves(ratio, T) < thr).mean()
                                         + (t_moves(1.0 / ratio, T) < thr).mean())
                den += r.weight        # renormalise if a pair has no data that day
            if den:
                rows_out.append({"day": pd.Timestamp(day), "group": g, "rate": num / den})
daily = pd.DataFrame(rows_out)

fig, ax = plt.subplots(figsize=(11, 4))
for g, sub in daily.groupby("group"):
    sub = sub.sort_values("day")
    ax.plot(sub.day, sub.rate, lw=1.1, color=COLORS[g],
            label=f"{g} (cap = {fit_caps[g]:.2f} bps)")
ax.axhline(TARGET_REVERT_RATE, color="gray", ls="--", lw=1,
           label=f"target = {TARGET_REVERT_RATE:.0%}")
ax.axvspan(pd.Timestamp(MONTHS[FIT_MONTH][0]), pd.Timestamp(MONTHS[FIT_MONTH][1]),
           color="C2", alpha=0.08, label=f"fit month ({FIT_MONTH})")
ax.set(ylabel="price-driven revert rate",
       title=f"deviation over time at the {FIT_MONTH}-fitted caps, T = {T}s")
ax.legend(fontsize=8)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"daily rate at the fitted cap, T = {T}s:")
print(daily.groupby("group").rate.agg(
    min="min", median="median", max="max",
    days_over_target=lambda s: int((s > TARGET_REVERT_RATE).sum())).to_string(float_format=lambda v: f"{v:7.2%}"))

In [ ]:
per_pair = pairs.copy()
per_pair["rate"] = [np.interp(fit_caps[r.group], CAP_GRID, pair_curve(r, FIT_MONTH, T))
                    for r in pairs.itertuples()]
per_pair = per_pair.sort_values("rate")

_, ax = plt.subplots(figsize=(7.5, 0.4 * len(per_pair) + 1))
ax.barh(range(len(per_pair)), per_pair.rate, color=per_pair.group.map(COLORS))
ax.set_yticks(range(len(per_pair)), per_pair.pair, fontsize=8)
ax.axvline(TARGET_REVERT_RATE, color="gray", ls="--", lw=1)
ax.set(xlabel=f"{FIT_MONTH} revert rate at the tier's fitted cap",
       title=f"deviation across token pairs, T = {T}s")
plt.tight_layout()
plt.show()

print(per_pair[["pair", "group", "attempts", "weight", "rate"]].to_string(
    index=False, float_format=lambda v: f"{v:8.3f}"))

## 3. Caveats

1. **Price-driven reverts only.** Realized total revert rates are higher, since they also include
   non-price failure modes.
2. **Moves are measured start-to-end of each window** — the price at the deadline. Scoring the
   worst point *within* the window instead raises the uncorrelated cap by ~22% at `T = 26s`.
3. **Tick resolution bounds the cap.** The correlated cap is one `USDCUSDT` tick (0.1 bps), so
   read it as [0, 0.2] bps; `cap(T)` flat in `T` is the signature. Small `T` is limited the same
   way for every pair. The correlated tier also covers EUR stables, ETH liquid-staking tokens and
   tokenised equities, whose cap needs differ; only USD-vs-USD is measured here.
4. **Coverage.** Only the pairs in `PAIRS` are measured. They are a minority of all settlement
   attempts; the unmapped tail is not represented.